[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/langgraph/langgraph_router.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# LangGraph 路由模式：按意图分配智能体
本教程使用 LangGraph 探索路由模式。LangGraph 是构建动态、有状态 LLM 应用的框架。我们将构建智能客服助手，根据用户意图将客户查询分配给账单、技术支持或常规信息专业智能体。

LangGraph 允许定义结构化逻辑图：路由节点对输入分类，条件边将其转发给正确的子智能体。每个智能体根据自己的上下文（如账单历史、故障排除提示）生成回复。

我们还使用 Phoenix 追踪应用，完整查看路由决策、工具使用和模型交互，便于调试路由准确性并理解图的端到端执行。

In [ ]:
!pip install langgraph langchain langchain_community "arize-phoenix" arize-phoenix-otel openinference-instrumentation-langchain

In [ ]:
!pip install langchain_openai

In [ ]:
import os
from getpass import getpass

from langgraph.graph import END, START, StateGraph

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

# 配置 Phoenix 追踪
请前往 https://app.phoenix.arize.com/ 生成 API 密钥，以便使用 Phoenix 追踪 LangGraph 应用。

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="Router", auto_instrument=True)

In [ ]:
from langchain_openai import ChatOpenAI

# LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [ ]:
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

# 路由器
使用 Pydantic 模式约束路由输出，确保 LLM 返回三个有效目标之一："billing"、"tech_support" 或 "general_info"。
再通过 with_structured_output 封装 LLM，使 LangGraph 在路由时强制采用这种结构化回复。

In [ ]:
class Route(BaseModel):
    step: Literal["billing", "tech_support", "general_info"] = Field(
        description="对支持请求分类"
    )


router = llm.with_structured_output(Route)

# 定义图状态
此状态模式记录路由请求的生命周期，保存原始用户输入（input）、路由器的分类决策（decision），以及对应支持处理器生成的最终回复（output）。图中各节点都会读写这一共享状态。

In [ ]:
class State(TypedDict):
    input: str
    decision: str
    output: str

# 客服智能体节点：专业回复处理器
本节定义三个由 LLM 驱动的专业智能体，分别处理不同类别的用户查询：

**账单智能体**：根据用户账单历史和账单政策上下文回答发票或退款问题。

**技术支持智能体**：利用预设支持知识库回答常见故障排除问题。

**常规信息智能体**：通过常见问答回答账号、订阅和政策问题。

每个智能体读取图状态中的 input 字段，返回符合其领域的 output。这些智能体构成路由图的执行终点。

In [ ]:
# 账单智能体
billing_history_data = [
    {"invoice_id": "INV001", "date": "2024-11-01", "amount": "$29.99"},
    {"invoice_id": "INV002", "date": "2024-12-01", "amount": "$29.99"},
    {"invoice_id": "INV003", "date": "2025-01-01", "amount": "$39.99"},
]

billing_general_context = (
    "Billing inquiries may include refunds, invoices, plan upgrades, or charges. "
    "Our system charges users monthly based on their plan. Refunds are processed within 5–7 business days."
)


def billing_agent(state: State):
    user_input = state["input"]
    invoice_summary = "\n".join(
        f"• Invoice {item['invoice_id']} on {item['date']} for {item['amount']}"
        for item in billing_history_data
    )
    prompt = (
        f"你是乐于助人的账单助手。\n"
        f"以下是用户的账单历史：\n{invoice_summary}\n\n"
        f"一般账单背景：\n{billing_general_context}\n\n"
        f"用户查询：\n{user_input}"
    )
    result = llm.invoke(prompt)
    return {"output": result.content}


# 技术支持智能体
tech_support_kb = (
    "Common issues include login errors, app crashes, and network connectivity. "
    "To fix login errors, check your email and reset your password. "
    "For crashes, try reinstalling the app. If your connection is unstable, restart your router."
)


def tech_support_agent(state: State):
    prompt = (
        "你是技术支持助手。请根据以下知识库回答用户问题。\n\n"
        f"知识库：\n{tech_support_kb}\n\n"
        f"用户查询：\n{state['input']}"
    )
    result = llm.invoke(prompt)
    return {"output": result.content}


# 常规信息智能体
general_info_kb = (
    "We offer 3 subscription plans: Basic, Pro, and Enterprise. "
    "Support is available 24/7. You can cancel your subscription any time from the account settings page."
)


def general_info_agent(state: State):
    prompt = (
        "你是常规信息助手。请根据以下知识库回答用户问题。\n\n"
        f"知识库：\n{general_info_kb}\n\n"
        f"用户查询：\n{state['input']}"
    )
    result = llm.invoke(prompt)
    return {"output": result.content}

# 意图分类与路由逻辑
本节介绍路由机制的决策逻辑：

**classify_intent 节点**：使用带结构化输出约束的 LLM，将查询分类为 billing、tech_support 或 general_info，并将结果存入状态的 decision 键。

**route_to_agent 函数**：根据分类结果将请求发送到对应专业智能体节点，使 LangGraph 能动态选择最相关的回复模块。

In [ ]:
def classify_intent(state: State):
    decision = router.invoke(
        [
            SystemMessage(
                content="将此支持请求分类为 billing、tech_support 或 general_info。"
            ),
            HumanMessage(content=state["input"]),
        ]
    )
    return {"decision": decision.step}


def route_to_agent(state: State):
    if state["decision"] == "billing":
        return "billing_agent"
    elif state["decision"] == "tech_support":
        return "tech_support_agent"
    elif state["decision"] == "general_info":
        return "general_info_agent"

# 构建路由图
通过注册节点并定义连接，构建完整 LangGraph 工作流：

**注册节点**：将 billing_agent、tech_support_agent、general_info_agent 和 classify_intent 加入图中。

**条件路由**：图从 classify_intent 开始，根据 route_to_agent 的输出，将查询转发给分类意图对应的智能体节点。

**终止边**：每个智能体节点直接连接 END，在回复查询后结束工作流。

这一设计体现了典型客服路由架构，可通过模块化方式扩展不同查询类型的处理。

In [ ]:
builder = StateGraph(State)

builder.add_node("classify_intent", classify_intent)
builder.add_node("billing_agent", billing_agent)
builder.add_node("tech_support_agent", tech_support_agent)
builder.add_node("general_info_agent", general_info_agent)

builder.add_edge(START, "classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_to_agent,
    {
        "billing_agent": "billing_agent",
        "tech_support_agent": "tech_support_agent",
        "general_info_agent": "general_info_agent",
    },
)

builder.add_edge("billing_agent", END)
builder.add_edge("tech_support_agent", END)
builder.add_edge("general_info_agent", END)

workflow = builder.compile()

# 运行一些查询：

In [ ]:
queries = [
    "Why was I charged $39.99 this month?",  # 账单
    "The app keeps crashing when I open it.",  # 技术支持
    "Can I cancel my subscription anytime?",  # 常规信息
    "Can you show me all my past invoices?",  # 账单
    "How do I fix login issues with my account?",  # 技术支持
]

# 将每条查询发送到路由器
for i, query in enumerate(queries, start=1):
    print(f"\n--- Query {i} ---")
    state = workflow.invoke({"input": query})
    print(f"Input: {query}")
    print(f"Response:\n{state['output']}")

# 记得在 Phoenix 中查看追踪！

# 评估

本节添加简单评估，检查路由器是否选中了正确的子智能体。子智能体路由类似选择要调用的工具，因此使用 Phoenix 的工具调用评估模板。

## 获取所有根跨度

In [ ]:
from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery

df = Client().spans.get_spans_dataframe(
    query=SpanQuery().where("name == 'LangGraph'"), project_name="Router"
)

In [ ]:
df.rename(columns={"attributes.output.value": "tool_call"}, inplace=True)
df.rename(columns={"attributes.input.value": "question"}, inplace=True)

In [ ]:
df_clean = df[["context.span_id", "question", "tool_call"]]
df_clean

## 将子智能体定义作为工具添加

每列都必须包含可选工具的信息。本例中的“工具”就是子智能体，因此传入各子智能体的定义（描述）。

In [ ]:
subagent_definitions = """
 billing: 根据用户账单历史与账单政策回答发票或退款相关问题。
tech_support: 使用预定义支持知识库回答常见故障排除查询。
general_info: 使用常见问答回答账号、订阅和政策相关问题。
"""
df_clean["tool_definitions"] = subagent_definitions

In [ ]:
df_clean

# 提取选中的子智能体。

In [ ]:
import re


def extract_decision(x):
    """使用正则表达式从字符串提取 decision 值。

    处理正则搜索返回 None 的情况。
    """
    if x is None:
        return None
    match = re.search(r'"decision"\s*:\s*"([^"]+)"', x)
    if match:
        return match.group(1)
    else:
        return None  # 也可使用你选择的默认值


df_clean["tool_call"] = df_clean["tool_call"].apply(extract_decision)
df_clean

In [ ]:
df_clean.to_csv("router_evals.csv", index=False)

# 运行评估

In [ ]:
from phoenix.evals import LLM

eval_llm = LLM(provider="openai", model="gpt-4o")

In [ ]:
from phoenix.evals import async_evaluate_dataframe, bind_evaluator
from phoenix.evals.metrics import ToolSelectionEvaluator

tool_selection_evaluator = ToolSelectionEvaluator(llm=eval_llm)

# 将 df_clean 的列映射到 ToolSelectionEvaluator 所需输入
bound_evaluator = bind_evaluator(
    evaluator=tool_selection_evaluator,
    input_mapping={
        "input": "question",
        "available_tools": "tool_definitions",
        "tool_selection": "tool_call",
    },
)

# 注意：此处不使用 ToolInvocationEvaluator，因为该路由模式
# 仅按名称选择子智能体，没有需要评估的工具参数。
response_classifications = await async_evaluate_dataframe(
    dataframe=df_clean,
    evaluators=[bound_evaluator],
    concurrency=10,
)

In [ ]:
response_classifications[["tool_selection_score", "tool_selection_execution_details"]]

# 将评估导出到 Phoenix！

In [ ]:
# async_evaluate_dataframe 输出的列已经整理完毕，无需删除多余列
pass

In [ ]:
response_classifications.to_parquet("router_evals.parquet")

In [ ]:
from phoenix.client import AsyncClient
from phoenix.evals.utils import to_annotation_dataframe

px_client = AsyncClient()

annotation_df = to_annotation_dataframe(response_classifications, ["tool_selection"])
await px_client.spans.log_span_annotations_dataframe(
    dataframe=annotation_df,
    annotator_kind="LLM",
)